In [ ]:
from google.colab import files
uploaded = files.upload()


Saving clientes_crm.json to clientes_crm.json


In [ ]:
import pandas as pd
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

In [ ]:
df_clientes = pd.read_json("clientes_crm.json")

print("Clientes carregado!")
print("Tamanho:", df_clientes.shape)

✅ Clientes carregado!
Tamanho: (49, 4)


In [ ]:
print("Colunas:", df_clientes.columns.tolist())
print("\nTipos de dados:")
print(df_clientes.dtypes)
print("\nPrimeiras linhas:")
df_clientes.head()

Colunas: ['full_name', 'location', 'code', 'email']

Tipos de dados:
full_name    object
location     object
code          int64
email        object
dtype: object

Primeiras linhas:


,full_name,location,code,email
0,Femininos Oliveira Antunes,"Aratu (Candeias) , BA",1,femininos.oliveira.antunes@icloud.com
1,Fernanda Azevedo Soares Nunes Vieira,"PE , Recife",2,nunes.fernanda.soares.azevedo.vieira@outlook.com
2,Daniel Farias Ribeiro Teixeira,"Rio Grande,RS",3,farias.teixeira.daniel.ribeiro@gmail.com
3,Thiago Moreira,"AC , Rio Branco",4,thiago.moreira@gmail.com
4,Pedro Freitas,PA - Santarém Novo,5,pedro.freitas@icloud.com


In [ ]:
# Ver exemplos do problema
print("Exemplos de e-mails com problema:")
print(df_clientes[df_clientes["email"].str.contains("#", na=False)]["email"].head(10))

# Corrigir substituindo # por @
df_clientes["email"] = df_clientes["email"].str.replace("#", "@", regex=False)

# Verificar se ainda sobrou algum problema
print("\nE-mails ainda com #:", df_clientes["email"].str.contains("#", na=False).sum())

Exemplos de e-mails com problema:
2        farias.teixeira.daniel.ribeiro#gmail.com
3                        thiago.moreira#gmail.com
4                        pedro.freitas#icloud.com
6     torres.barros.rocha.bianca.siqueira#aol.com
7                 pimentel.alves.luiz#outlook.com
8           lucas.lopes.guedes.cunha#tutanota.com
9                            paiva.débora#gmx.com
11                 rafael.pereira.barros#zoho.com
12           martins.guimarães.carlos#hotmail.com
14      lopes.alves.pacheco.rocha.carla#yahoo.com
Name: email, dtype: object

E-mails ainda com #: 0
✅ E-mails corrigidos!


In [ ]:
print("Exemplos de location:")
print(df_clientes["location"].value_counts().head(90))

Exemplos de location:
location
BA - Porto Seguro                 2
Aratu (Candeias) , BA             1
Rio Grande,RS                     1
PE , Recife                       1
PA - Santarém Novo                1
Fortaleza do Tabocão , TO         1
PB/Cabedelo                       1
AC , Rio Branco                   1
SE - Aracaju                      1
PB - João Pessoa                  1
Santarém / PA                     1
TO , Fortaleza do Tabocão         1
PA / Santarém                     1
AM , Itacoatiara                  1
Fortaleza do Tabocão,TO           1
Fortaleza,CE                      1
MS - Corumbá                      1
Santarém - PA                     1
Maceió / AL                       1
PA , Santarém Novo                1
AC,Rio Branco                     1
SE / Aracaju                      1
Santos - SP                       1
Laguna / SC                       1
ES / São Mateus                   1
Manaus/AM                         1
Salvador,BA                      

In [ ]:
estados = {
    "AC", "AL", "AP", "AM", "BA", "CE", "DF", "ES", "GO",
    "MA", "MT", "MS", "MG", "PA", "PB", "PR", "PE", "PI",
    "RJ", "RN", "RS", "RO", "RR", "SC", "SP", "SE", "TO"
}

In [ ]:
import re

def extrair_cidade_estado(location):
    if pd.isna(location):
        return None, None

    # Limpar espaços extras
    location = str(location).strip()

    # Separar por , ou - ou /
    partes = re.split(r"[,\-/]", location)
    partes = [p.strip() for p in partes if p.strip()]

    cidade = None
    estado = None

    for parte in partes:
        parte_upper = parte.upper().strip()
        if parte_upper in estados:
            estado = parte_upper
        else:
            cidade = parte.title().strip()

    return cidade, estado

# Aplicar a função
df_clientes[["cidade", "estado"]] = df_clientes["location"].apply(
    lambda x: pd.Series(extrair_cidade_estado(x))
)

print(df_clientes[["location", "cidade", "estado"]].head(200))

✅ Separação concluída!
                          location                     cidade estado
0            Aratu (Candeias) , BA           Aratu (Candeias)     BA
1                      PE , Recife                     Recife     PE
2                    Rio Grande,RS                 Rio Grande     RS
3                  AC , Rio Branco                 Rio Branco     AC
4               PA - Santarém Novo              Santarém Novo     PA
5        Fortaleza do Tabocão , TO       Fortaleza Do Tabocão     TO
6                      PB/Cabedelo                   Cabedelo     PB
7                     SE - Aracaju                    Aracaju     SE
8                 PB - João Pessoa                João Pessoa     PB
9                    Santarém / PA                   Santarém     PA
10               BA - Porto Seguro               Porto Seguro     BA
11       TO , Fortaleza do Tabocão       Fortaleza Do Tabocão     TO
12                   PA / Santarém                   Santarém     PA
13         

In [ ]:
# Ver registros onde cidade ou estado ficaram nulos
problemas = df_clientes[df_clientes["estado"].isna() | df_clientes["cidade"].isna()]
print(f"Registros com problema: {len(problemas)}")
print(problemas[["location", "cidade", "estado"]])

Registros com problema: 0
Empty DataFrame
Columns: [location, cidade, estado]
Index: []


In [ ]:
correcoes_manual = {
    "Aratu (Candeias) , BA": ("Candeias", "BA"),
    "PB/Cabedelo": ("Cabedelo", "PB"),
    "PA - Santarém Novo": ("Santarém Novo", "PA"),
    "RJ - Petrópolis": ("Petrópolis", "RJ"),
}

for location, (cidade, estado) in correcoes_manual.items():
    mask = df_clientes["location"] == location
    df_clientes.loc[mask, "cidade"] = cidade
    df_clientes.loc[mask, "estado"] = estado

✅ Correções manuais aplicadas!


In [ ]:
print("Distribuição por estado:")
print(df_clientes["estado"].value_counts())

print("\nNulos restantes:")
print(df_clientes[["cidade", "estado"]].isnull().sum())

Distribuição por estado:
estado
PA    8
BA    5
TO    4
CE    3
PE    3
PB    3
SE    3
MA    3
AM    3
RS    2
AC    2
MS    2
PR    2
AL    1
SP    1
ES    1
SC    1
AP    1
RJ    1
Name: count, dtype: int64

Nulos restantes:
cidade    0
estado    0
dtype: int64


In [ ]:
# Verificar o que não foi identificado
problemas = df_clientes[df_clientes["estado"].isna() | df_clientes["cidade"].isna()]
print(f"Registros com problema: {len(problemas)}")
print(problemas[["location", "cidade", "estado"]])

Registros com problema: 0
Empty DataFrame
Columns: [location, cidade, estado]
Index: []


In [ ]:
print("Distribuição por estado:")
print(df_clientes["estado"].value_counts())

Distribuição por estado:
estado
PA    8
BA    5
TO    4
CE    3
PE    3
PB    3
SE    3
MA    3
AM    3
RS    2
AC    2
MS    2
PR    2
AL    1
SP    1
ES    1
SC    1
AP    1
RJ    1
Name: count, dtype: int64


In [ ]:
import re

estados = {
    "AC", "AL", "AP", "AM", "BA", "CE", "DF", "ES", "GO",
    "MA", "MT", "MS", "MG", "PA", "PB", "PR", "PE", "PI",
    "RJ", "RN", "RS", "RO", "RR", "SC", "SP", "SE", "TO"
}

def extrair_cidade_estado(location):
    if pd.isna(location):
        return None, None

    location = str(location).strip()

    # Remover conteúdo entre parênteses e usar como cidade
    location = re.sub(r"\(.*?\)", "", location).strip()

    # Separar por , ou - ou /
    partes = re.split(r"[,\-/]", location)
    partes = [p.strip() for p in partes if p.strip()]

    cidade = None
    estado = None

    for parte in partes:
        parte_upper = parte.upper().strip()
        if parte_upper in estados:
            estado = parte_upper
        else:
            if parte.strip():
                cidade = parte.title().strip()

    return cidade, estado

# Aplicar
df_clientes[["cidade", "estado"]] = df_clientes["location"].apply(
    lambda x: pd.Series(extrair_cidade_estado(x))
)

print(df_clientes[["location", "cidade", "estado"]].head(20))

✅ Separação concluída!
                     location                cidade estado
0       Aratu (Candeias) , BA                 Aratu     BA
1                 PE , Recife                Recife     PE
2               Rio Grande,RS            Rio Grande     RS
3             AC , Rio Branco            Rio Branco     AC
4          PA - Santarém Novo         Santarém Novo     PA
5   Fortaleza do Tabocão , TO  Fortaleza Do Tabocão     TO
6                 PB/Cabedelo              Cabedelo     PB
7                SE - Aracaju               Aracaju     SE
8            PB - João Pessoa           João Pessoa     PB
9               Santarém / PA              Santarém     PA
10          BA - Porto Seguro          Porto Seguro     BA
11  TO , Fortaleza do Tabocão  Fortaleza Do Tabocão     TO
12              PA / Santarém              Santarém     PA
13           AM , Itacoatiara           Itacoatiara     AM
14    Fortaleza do Tabocão,TO  Fortaleza Do Tabocão     TO
15               Fortaleza,CE    

In [ ]:
print("Colunas:", df_clientes.columns.tolist())
print("\nTipos de dados:")
print(df_clientes.dtypes)
print("\nPrimeiras linhas:")
df_clientes.head()

Colunas: ['full_name', 'location', 'code', 'email', 'cidade', 'estado']

Tipos de dados:
full_name    object
location     object
code          int64
email        object
cidade       object
estado       object
dtype: object

Primeiras linhas:


,full_name,location,code,email,cidade,estado
0,Femininos Oliveira Antunes,"Aratu (Candeias) , BA",1,femininos.oliveira.antunes@icloud.com,Aratu,BA
1,Fernanda Azevedo Soares Nunes Vieira,"PE , Recife",2,nunes.fernanda.soares.azevedo.vieira@outlook.com,Recife,PE
2,Daniel Farias Ribeiro Teixeira,"Rio Grande,RS",3,farias.teixeira.daniel.ribeiro@gmail.com,Rio Grande,RS
3,Thiago Moreira,"AC , Rio Branco",4,thiago.moreira@gmail.com,Rio Branco,AC
4,Pedro Freitas,PA - Santarém Novo,5,pedro.freitas@icloud.com,Santarém Novo,PA


In [ ]:
print("Colunas:", df_clientes.columns.tolist())
print("\nNulos por coluna:")
print(df_clientes.isnull().sum())
print("\nDuplicatas:", df_clientes.duplicated().sum())

Colunas: ['full_name', 'location', 'code', 'email', 'cidade', 'estado']

Nulos por coluna:
full_name    0
location     0
code         0
email        0
cidade       0
estado       0
dtype: int64

Duplicatas: 0


In [ ]:
# Padronizar nomes
df_clientes["full_name"] = df_clientes["full_name"].str.strip().str.title()

print(df_clientes["full_name"].head(50))

✅ full_name tratado!
0                      Femininos Oliveira Antunes
1            Fernanda Azevedo Soares Nunes Vieira
2                  Daniel Farias Ribeiro Teixeira
3                                  Thiago Moreira
4                                   Pedro Freitas
5      Antônia Coelho Pinheiro Peixoto Cavalcanti
6             Bianca Barros Rocha Torres Siqueira
7                             Luiz Alves Pimentel
8                        Lucas Guedes Cunha Lopes
9                                    Débora Paiva
10                         Victor Torres Monteiro
11                          Rafael Pereira Barros
12                       Carlos Guimarães Martins
13                   Gabriela Silva Vieira Amaral
14                Carla Lopes Alves Pacheco Rocha
15                         Renata Pacheco Cardoso
16                Luís Paiva Costa Cardoso Coelho
17              Adriana Guedes Borges Alves Rocha
18              Renata Lima Gomes Coelho Mendonça
19                           

In [ ]:
# Corrigir # por @
df_clientes["email"] = df_clientes["email"].str.replace("#", "@", regex=False)

# Padronizar para minúsculas
df_clientes["email"] = df_clientes["email"].str.strip().str.lower()

# Verificar se ainda tem problemas
emails_invalidos = df_clientes[~df_clientes["email"].str.contains("@", na=False)]
print(f"E-mails inválidos restantes: {len(emails_invalidos)}")

E-mails inválidos restantes: 0
✅ E-mails tratados!


In [ ]:
print("Códigos únicos:", df_clientes["code"].nunique())
print("Total de registros:", len(df_clientes))

df_clientes["code"] = df_clientes["code"].astype(str).str.strip()

Códigos únicos: 49
Total de registros: 49
✅ code tratado!


In [ ]:
df_clientes["cidade"] = df_clientes["cidade"].str.strip().str.title()
df_clientes["estado"] = df_clientes["estado"].str.strip().str.upper()

# Verificar estados válidos
estados_validos = {
    "AC", "AL", "AP", "AM", "BA", "CE", "DF", "ES", "GO",
    "MA", "MT", "MS", "MG", "PA", "PB", "PR", "PE", "PI",
    "RJ", "RN", "RS", "RO", "RR", "SC", "SP", "SE", "TO"
}

estados_invalidos = df_clientes[~df_clientes["estado"].isin(estados_validos)]
print(f"Estados inválidos: {len(estados_invalidos)}")

Estados inválidos: 0
✅ cidade e estado tratados!


In [ ]:
# Agora que temos cidade e estado separados, a coluna original não é mais necessária
df_clientes = df_clientes.drop(columns=["location"])

print("Colunas finais:", df_clientes.columns.tolist())

✅ Coluna location removida!
Colunas finais: ['full_name', 'code', 'email', 'cidade', 'estado']


In [ ]:
df_clientes.to_csv("clientes_limpo.csv", index=False)

✅ Base de clientes limpa salva!
